In [32]:
from pyspark.sql import SparkSession

spark= SparkSession.builder.appName("Online Retail").getOrCreate()

In [33]:
df= spark.read.csv("Online_Retail.csv",header=True,inferSchema=True)
df.printSchema()
df.show(5)
print("Row Count:", df.count())

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity| InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/10 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/10 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/10 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|       6|12/1/

In [34]:
from pyspark.sql.functions import col

df_clean= df.filter(
    (col('Quantity')>0) &
    (col("UnitPrice")>0) &
    (col("InvoiceDate").isNotNull()) &
    (col("CustomerID").isNotNull())
)

print("Clean Rows", df_clean.count())


Clean Rows 397884


In [36]:
df_clean.select("InvoiceDate").show(5,truncate=False)

+------------+
|InvoiceDate |
+------------+
|12/1/10 8:26|
|12/1/10 8:26|
|12/1/10 8:26|
|12/1/10 8:26|
|12/1/10 8:26|
+------------+
only showing top 5 rows


In [47]:
from pyspark.sql.functions import to_timestamp,month,dayofweek,hour,round as spark_round
df_clean = df_clean.withColumn('TotalPrice',spark_round((col("UnitPrice") * col("Quantity")),2))
df_clean = df_clean.withColumn('InvoiceDate',to_timestamp(col("InvoiceDate"),"M/d/yy H:mm"))
df_clean = df_clean.withColumn('Month', month(col('InvoiceDate')))
df_clean = df_clean.withColumn('DayOfWeek',dayofweek(col('InvoiceDate')))

df_clean.show(5)


+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+----------+-----+---------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|TotalPrice|Month|DayOfWeek|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+----------+-----+---------+
|   536365|   85123A|WHITE HANGING HEA...|       6|2010-12-01 08:26:00|     2.55|     17850|United Kingdom|      15.3|   12|        4|
|   536365|    71053| WHITE METAL LANTERN|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|     20.34|   12|        4|
|   536365|   84406B|CREAM CUPID HEART...|       8|2010-12-01 08:26:00|     2.75|     17850|United Kingdom|      22.0|   12|        4|
|   536365|   84029G|KNITTED UNION FLA...|       6|2010-12-01 08:26:00|     3.39|     17850|United Kingdom|     20.34|   12|        4|
|   536365|   84029E|RED WOOLLY HOTTIE...|       6|2010

In [48]:
df_clean.groupBy("Country") \
        .sum('TotalPrice') \
        .withColumnRenamed('sum(TotalPrice)','Revenue') \
        .orderBy(col('Revenue').desc()) \
        .show()

+---------------+------------------+
|        Country|           Revenue|
+---------------+------------------+
| United Kingdom| 7308391.550000088|
|    Netherlands|         285446.34|
|           EIRE|265545.90000000026|
|        Germany|228867.13999999996|
|         France|209024.04999999996|
|      Australia|         138521.31|
|          Spain|          61577.11|
|    Switzerland|56443.949999999975|
|        Belgium| 41196.34000000002|
|         Sweden|          38378.33|
|          Japan|          37416.37|
|         Norway|          36165.44|
|       Portugal|          33439.89|
|        Finland|          22546.08|
|      Singapore|          21279.29|
|Channel Islands| 20450.44000000001|
|        Denmark|          18955.34|
|          Italy|17483.239999999998|
|         Cyprus|          13590.38|
|        Austria|          10198.68|
+---------------+------------------+
only showing top 20 rows


In [49]:
df_clean.groupBy('Month') \
    .sum('TotalPrice') \
    .withColumnRenamed("sum(TotalPrice)","Revenue") \
    .orderBy('Month').show()

+-----+------------------+
|Month|           Revenue|
+-----+------------------+
|    1| 569445.0400000141|
|    2| 447137.3500000145|
|    3|  595500.760000013|
|    4| 469200.3600000017|
|    5| 678594.5600000031|
|    6| 661213.6900000116|
|    7| 600091.0100000063|
|    8| 645343.9000000106|
|    9| 952838.3799999992|
|   10|1039318.7899999996|
|   11|  1161817.38000004|
|   12| 1090906.680000019|
+-----+------------------+



In [ ]:
df_clean.groupBy('StockCode') \
        .sum('Quantity','TotalPrice') \
        .withColumnRenamed('sum(Quantity)','TotalQuantity') \
        .withColumnRenamed('sum(TotalPrice)','Revenue') \
        .orderBy(col('Revenue').desc()) \
        .show()

        

PySparkAttributeError: [ATTRIBUTE_NOT_SUPPORTED] Attribute `desc` is not supported.